# 영상 기반 면접 피드백 (멀티모달)

면접 영상을 넣으면 Gemini가 자동으로 분석해서 피드백을 생성합니다.

## 데이터 역할
| 데이터 | 역할 |
|---|---|
| `data/feedback_data/` | **Few-Shot** — 이형 어투/스타일 학습 |
| `data/rag_data/` (ChromaDB) | **RAG** — 이형 강의 지식/논리 근거 |
| 면접 영상 (.mp4) | **입력** — Gemini가 영상에서 직접 음성+비언어 분석 |

## Gemini가 영상에서 자동으로 처리하는 것
- 🎤 음성 → 답변 내용 파악 (STT 별도 불필요)
- 👁️ 시선, 표정, 고개 방향
- 🤝 제스처, 자세
- 🕐 타임스탬프 `[MM:SS]` 기반 피드백

## 전제 조건
- `01_rag_build.ipynb` 먼저 실행 → `chroma_db/` 생성 필요
- 영상 파일 경로 준비

In [ ]:
%pip install google-genai chromadb==0.6.3 sentence-transformers python-dotenv --quiet

---
## STEP 1 — 환경 설정

In [ ]:
import os, re, time, random
from pathlib import Path
from dotenv import load_dotenv

BASE       = Path(r'C:\Users\82105\OneDrive\바탕 화면\interview-coach')
PROJ       = BASE.parent / '프로젝트3(면접)'
FB_DIR     = BASE / 'data' / 'feedback_data'   # Few-Shot용
CHROMA_DIR = PROJ / 'chroma_db'                # RAG DB

load_dotenv(PROJ / '.env')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
if not GOOGLE_API_KEY:
    raise ValueError('.env 파일에 GOOGLE_API_KEY가 없습니다.')

# 발표자료 기준 모델. 단 무료 등급은 쿼터 0이라 결제 활성화 필요
# (무료로 실행하려면 아래 FALLBACK 줄로 교체 — 영상 멀티모달 지원 모델이어야 함)
MODEL = 'gemini-3.1-pro-preview'
# MODEL = 'gemini-2.5-flash'   # FALLBACK: 무료 등급 사용 가능

print(f'설정 완료 | 모델: {MODEL}')

---
## STEP 2 — RAG 로드

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = chroma_client.get_collection('interview_rag', embedding_function=ef)
print(f'RAG 로드 완료: {collection.count()}개 청크')

def retrieve(query: str, n: int = 4):
    results = collection.query(query_texts=[query[:200]], n_results=n)
    return results['documents'][0]

---
## STEP 3 — Few-Shot 예시 로드 (feedback_data)

In [ ]:
pairs = []

for iv_f in sorted((FB_DIR / '1~5' / 'Interview_text').glob('*.txt')):
    fb_f = FB_DIR / '1~5' / 'Feedback_text' / iv_f.name
    if fb_f.exists():
        iv = iv_f.read_text(encoding='utf-8').strip()
        fb = fb_f.read_text(encoding='utf-8').strip()
        if iv and fb:
            pairs.append({'interview': iv, 'feedback': fb})

for iv_f in sorted((FB_DIR / '11~16' / 'interview_text').glob('*.txt')):
    fb_f = FB_DIR / '11~16' / 'feedback_text' / iv_f.name
    if fb_f.exists():
        iv = iv_f.read_text(encoding='utf-8').strip()
        fb = fb_f.read_text(encoding='utf-8').strip()
        if iv and fb:
            pairs.append({'interview': iv, 'feedback': fb})

text_dir = FB_DIR / '6~10' / 'text'
for iv_f in sorted(text_dir.glob('*인터뷰*.txt')):
    num = re.match(r'(\d+)', iv_f.name)
    if not num:
        continue
    n = num.group(1)
    iv = iv_f.read_text(encoding='utf-8').strip()
    fb = '\n\n'.join(f.read_text(encoding='utf-8').strip()
                     for f in sorted(text_dir.glob(f'{n}_[0-9]*.txt')))
    if iv and fb:
        pairs.append({'interview': iv, 'feedback': fb})

print(f'Few-Shot 예시: {len(pairs)}개 답변-피드백 쌍')

---
## STEP 4 — Gemini 클라이언트 + 프롬프트

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

# 발표자료 slide19 최종 프롬프트 (타임스탬프 + 비언어 규칙 포함)
SYSTEM_PROMPT = """대기업 인사 전문가이자 커리어 코칭 전문가인 '인터뷰 킹 이형' 스타일의 면접 피드백 코치입니다.
말투는 친절할 수 있지만 평가는 차갑고 직설적이어야 합니다.
학습한 내용을 바탕으로 지원자의 답변에 대한 피드백을 출력합니다.

중요한 규칙:
- 후보자의 답변에만 피드백을 제공합니다.
- 후속 질문, 예상 질문 또는 추가 질문 목록을 생성하지 마십시오.
- 일반적이거나 진부하거나 '상식적인' 조언은 피하세요.
- 후보자의 답변에 제공된 실제 내용을 바탕으로 평가합니다.
- 약점을 지적할 때, 왜 그것들이 부족한지 그리고 실제 면접관이 그 특정 점을 어떻게 인식하는지 설명하세요.
- 강점은 진정으로 얻은 것일 때만 인정하고, 지나치게 칭찬하지 마세요.
- 학습된 내용을 기반으로 하되, 학습된 내용은 참고하는 형식으로만 하고 너무 똑같이 피드백하지 마세요.
- 학습된 내용에서 중요하게 판단하는 내용들을 종합해서 피드백하도록 합니다.
- 제공된 비디오를 분석할 때, 반드시 [MM:SS] 형식의 타임스탬프를 붙여서 피드백하라.
  특히 답변 내용과 비언어적 태도가 불일치하는 지점의 시간을 정확히 명시하라.

출력 형식:
- 섹션 제목이나 제목 없이 자연스럽고 대화적인 흐름으로 작성하세요.
- 첫 번째 문장부터 바로 평가를 시작하세요.
- 후보자의 답변 중 개선해야 할 부분에 대해 구체적이고 실행 가능한 방향으로 결론을 내립니다.
- 모든 답변은 한국어로 작성해야 합니다."""

print('프롬프트 설정 완료')

---
## STEP 5 — 영상 업로드 & 피드백 생성

> **VIDEO_PATH를 분석할 영상 파일 경로로 수정하세요.**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VIDEO_PATH = r'C:\Users\82105\OneDrive\바탕 화면\drive_unzip\최종테스트_영상\면접영상.mp4'

N_SHOTS = 5   # Few-Shot 예시 수
N_RAG   = 4   # RAG 청크 수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

video_path = Path(VIDEO_PATH)
print(f'영상: {video_path.name}  ({video_path.stat().st_size // 1024 // 1024} MB)')

# 1. 영상 업로드 (Gemini File API)
print('업로드 중...')
video_file = client.files.upload(
    path=str(video_path),
    config={'mime_type': 'video/mp4'}
)
print(f'업로드 완료: {video_file.name}')

# 처리 완료 대기
print('Gemini 처리 대기 중...')
while video_file.state.name == 'PROCESSING':
    time.sleep(5)
    video_file = client.files.get(name=video_file.name)
    print(f'  상태: {video_file.state.name}', end='\r')

if video_file.state.name == 'FAILED':
    raise RuntimeError(f'영상 처리 실패: {video_file.state}')

print(f'\n처리 완료! 상태: {video_file.state.name}')

In [ ]:
# 2. RAG 검색 — 영상 내용 키워드로 검색
#    (영상 내용을 먼저 파악하는 사전 호출)
pre_resp = client.models.generate_content(
    model=MODEL,
    contents=[video_file, '이 영상에서 지원자가 말하는 핵심 키워드를 5개만 추출해줘. 단어만.'],
    config=types.GenerateContentConfig(max_output_tokens=100)
)
keywords = pre_resp.text
print(f'영상 키워드: {keywords}')

rag_docs = retrieve(keywords, n=N_RAG)
rag_context = '\n\n'.join(f'[참고{i+1}] {doc}' for i, doc in enumerate(rag_docs))
print(f'RAG 검색 완료: {len(rag_docs)}개 청크')

In [ ]:
# 3. Few-Shot contents 구성 (Q./A. 형식)
shots = random.sample(pairs, min(N_SHOTS, len(pairs)))
contents = []

for shot in shots:
    contents.append(types.Content(
        role='user',
        parts=[types.Part.from_text(text=f'Q.\n{shot["interview"]}')]
    ))
    contents.append(types.Content(
        role='model',
        parts=[types.Part.from_text(text=f'A.\n{shot["feedback"]}')]
    ))

# 4. 실제 질문 — 영상 파일 + RAG 참고자료
contents.append(types.Content(
    role='user',
    parts=[
        types.Part.from_uri(file_uri=video_file.uri, mime_type='video/mp4'),
        types.Part.from_text(
            text=f'Q.\n위 면접 영상을 분석해줘.\n\n'
                 f'--- 참고 자료 (면접왕 이형 강의) ---\n{rag_context}'
        )
    ]
))

# 5. Gemini 호출
print(f'Few-Shot {N_SHOTS}개 + RAG {N_RAG}개 + 영상 → Gemini 호출 중...')

resp = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.4,
        max_output_tokens=1500
    )
)

print('\n' + '━' * 60)
print(resp.text)
print('━' * 60)

---
## STEP 6 — 업로드 파일 정리 (선택)

In [ ]:
# Gemini File API에 올라간 파일 목록 확인 및 삭제
print('업로드된 파일 목록:')
for f in client.files.list():
    size_mb = getattr(f, 'size_bytes', 0) // 1024 // 1024
    print(f'  {f.name}  ({size_mb} MB)  상태: {f.state.name}')

# 삭제하려면 아래 주석 해제
# client.files.delete(name=video_file.name)
# print('삭제 완료')